# N079 · 图的表示与建模

**目标：** 从关系、网格或依赖中明确顶点、边和方向。

**先修：** N018, N007, N053。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 有向/无向；邻接表/矩阵/边表；度；隐式图；自环与重边。

**配套讲解来源：** [同名逐章意见](../../comment/079_graph_representations.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

算法题里很少直接给你"一个图"，更多时候给你的是一堆关系：朋友对、道路双向连接、课程先后依赖、网格上的可移动规则。你要做的第一步是把这些关系翻译成代码能用的数据结构——这一步叫**建模**：先想清楚"谁是点、谁是边、边有没有方向"，再决定用邻接表、邻接矩阵还是边列表来存。

本章落到三个具体问题上：

1. **建邻接表：** 给你 n 个顶点（编号 0..n-1）和一些边，把它们存成"每个点连着谁"的列表。具体例子：n=4、边 `[(0,1),(1,2)]`、无向，结果应是 `[[1],[0,2],[],[]]`——点 0 连着 1，点 1 连着 0 和 2，点 2 连着 1，点 3 谁也不连但**必须保留**（它是孤立点）。
2. **邻接矩阵转边表：** 给你一个 n×n 的 0/1 矩阵，matrix[i][j] 非 0 表示 i 到 j 有边，把它转成边列表。无向图的矩阵是对称的，若直接全部输出每条边会出现两次（(0,1) 和 (1,0) 是同一条边），所以只取上三角。例子：`[[0,1,0],[1,0,1],[0,1,0]]` → `[(0,1),(1,2)]`，而不是四条边。
3. **找法官：** 小镇 n 个人（编号 1..n），trust 里每条 `[a,b]` 表示 a 信任 b。法官要满足两条：其他所有人（n-1 个）都信任他，且他不信任任何人。输入 n=2、trust=[[1,2]] → 输出 2，因为 1 信任 2，且 2 没信任过任何人，1 号被信任 0 次不达标。若 trust=[[1,3],[2,3],[3,1]]，3 虽然被 1、2 信任，但他信任了 1（出度不为 0），所以输出 -1。这个题的关键洞察是：**根本不用搜图，数每个点的入度和出度就够了。**

## 2. 基础知识：先读懂这些词

- **顶点（vertex）与边（edge）：** 顶点是研究对象（人、城市、格子），边是两两之间的关系。建模的第一问永远是：这个题里"点"是什么、"边"是什么？
- **有向图 / 无向图：** 边带箭头（a→b 只能单向走）还是有边就算互通。无向边在存储时通常要**存两份**（a 的表里加 b，b 的表里加 a），这是新手最常忘的一步。
- **邻接表（adjacency list）：** "每个点 → 它的邻居列表"。适合**稀疏图**（边数远小于点数的平方），空间 O(V+E)。一条边只占两个列表位置，省内存。
- **邻接矩阵（adjacency matrix）：** n×n 的二维表，matrix[i][j] 存 i→j 的边权或 0/1。判断"i 和 j 有没有边"只要 O(1)，但空间是 O(V²)，稀疏图上浪费严重。无向图的邻接矩阵必须沿对角线对称。
- **边表（edge list）：** 直接存 `[(u,v),...]` 的列表。最朴素，适合一次遍历所有边的场景（比如前面章节的 Bellman-Ford）。
- **度（degree）：** 一个点连了几条边。无向图里就是邻居列表长度；有向图分**入度**（多少人指向我）和**出度**（我指向多少人）。法官问题的本质就是入度 = n-1 且出度 = 0。
- **孤立顶点：** 一条边都不沾的点。邻接表必须先为所有 0..n-1 各建一个空列表，再加边，否则孤立点会"消失"，后面数连通分量就会数错。
- **自环（self-loop）与重边：** 自环是 (a,a)，指回自己的边；重边是同两点间有多条边。代码里 `u!=v` 的判断就是在处理无向自环——`adj[u].append(v)` 之后如果再做 `adj[v].append(u)` 就等于把自环记了两遍。
- **隐式图：** 有些题不给你边，只给移动规则（比如国际象棋马走"日"字），点和边是由规则**隐含**定义的——练习 1 会让你亲手把这种规则变成边表。

## 3. 思路推导：从小例子开始

**Step 1：确定顶点编号范围。** 本章统一用 0..n-1（法官题例外，是 1..n，跟着题目走）。这一步决定了开多大的数组。

**Step 2：为每个顶点先建空槽，再加边。** `adj=[[] for _ in range(n)]` 先保证 n 个点都在，然后逐条边 append。无向边加两次（u 里加 v、v 里加 u）；有向边只加一次（u 里加 v）。这样孤立点自然保留。

**Step 3：矩阵转边表时先验证再输出。** 无向模式下矩阵必须对称，不对称说明输入本身有矛盾，直接报错；输出时用 `i<=j` 只扫上三角（对角线含自环也要保留），避免一条无向边输出两遍。

**Step 4：法官问题转化为数数。** 不需要建图搜索：扫一遍 trust，给每条 [a,b] 记 outgoing[a]+=1、incoming[b]+=1，然后找那个 incoming==n-1 且 outgoing==0 的人。

**手算演示（build_adj_list(4, [(0,1),(1,2)], False)）：**

- 初始：`adj=[[],[],[],[]]`（4 个点全都在，包括暂时孤零零的 2 和 3）。
- 处理边 (0,1)：无向，所以 `adj[0]` 追加 1，`adj[1]` 追加 0 → `[[1],[0],[],[]]`。
- 处理边 (1,2)：`adj[1]` 追加 2，`adj[2]` 追加 1 → `[[1],[0,2],[],[]]`。
- 最终 notebook 小实例表格展示的就是：点 0 度数 1、点 1 度数 2、点 2 度数 1、点 3 度数 0（孤立点没丢）。

**手算演示（find_judge(3, [[1,3],[2,3],[3,1]])）：**

- incoming 初始 [0,0,0,0]（下标 0 空着不用，因为人是 1..3）。
- [1,3]：outgoing[1]=1，incoming[3]=1。[2,3]：outgoing[2]=1，incoming[3]=2。[3,1]：outgoing[3]=1，incoming[1]=1。
- 逐个检查：1 号 incoming=1≠2；2 号 incoming=0≠2；3 号 incoming=2=n-1 达标，但 outgoing=1≠0，出局。没人达标 → -1。对照 find_judge(2,[[1,2]])：incoming[2]=1=n-1 且 outgoing[2]=0 → 答案 2。

## 4. 核心代码：build_adj_list

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def build_adj_list(n, edges, directed=False):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        if not (0 <= u < n and 0 <= v < n):
            raise ValueError('vertex outside 0..n-1')
        adj[u].append(v)
        if not directed and u != v:
            adj[v].append(u)
    return adj
```

### 逐段读懂代码

**第一段：build_adj_list——建邻接表。**

```python
def build_adj_list(n, edges, directed=False):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        if not (0 <= u < n and 0 <= v < n):
            raise ValueError('vertex outside 0..n-1')
        adj[u].append(v)
        if not directed and u != v:
            adj[v].append(u)
    return adj
```

- `directed=False` 是默认参数：不传就当无向图处理，这是最常见情况。
- 第一行先给每个点建空列表。注意不能写 `adj=[[]]*n`——那是 n 个指向**同一个**列表的引用，加一条边会同时出现在所有点的邻居里，是经典 Python 大坑。
- `if not (0<=u<n and 0<=v<n)`：链式比较 `0<=u<n` 等价于 `0<=u and u<n`。整句在挡**越界顶点**——比如 n=4 却来了边 (0,7)，与其让程序后面莫名其妙 IndexError，不如当场报清晰错误。
- `adj[u].append(v)` 之后那句是本章精髓：无向且不是自环时才反向补一条。`not directed and u!=v` 拆开就是"如果是无向图，而且 u 和 v 不是同一个点"。自环 (a,a) 只 append 一次，避免同一条自环记两遍。

**第二段：adj_matrix_to_edges——矩阵转边表。**

```python
def adj_matrix_to_edges(matrix, directed=False):
    n = len(matrix)
    if any((len(row) != n for row in matrix)):
        raise ValueError('square matrix required')
    if not directed and any((bool(matrix[i][j]) != bool(matrix[j][i]) for i in range(n) for j in range(n))):
        raise ValueError('undirected adjacency matrix must be symmetric')
    return [(i, j) for i in range(n) for j in range(n) if matrix[i][j] and (directed or i <= j)]
```

- 第一句校验矩阵是方阵（每行长度都等于行数），非方阵无从谈邻接矩阵。
- 第二句校验对称性：`bool(matrix[i][j])!=bool(matrix[j][i])` 把值统一当真假比，只要 (i,j) 和 (j,i) 一个有边一个没边就报错。双重 for 写在生成器里逐格扫。
- 最后的列表推导是三重条件过滤：`matrix[i][j]` 非 0（有边）、且（有向模式随意 / 无向模式只留 `i<=j` 即上三角含对角线）。对角线 `i==j` 不排除，是为了保留自环信息。

**第三段：find_judge——数入度出度找法官。**

```python
def find_judge(n, trust):
    incoming = [0] * (n + 1)
    outgoing = [0] * (n + 1)
    for u, v in set(map(tuple, trust)):
        outgoing[u] += 1
        incoming[v] += 1
    return next((i for i in range(1, n + 1) if incoming[i] == n - 1 and outgoing[i] == 0), -1)
```

- 数组开 `n+1` 长度，是为了让下标 1..n 直接对应人的编号，下标 0 闲置不用——这是和前两个函数（用 0..n-1）不一样的地方，notebook 的 “正确性与复杂度”部分 特意提醒了这个差异。
- `set(map(tuple,trust))` 先把 trust 去重：同一个 [a,b] 信任关系说两遍不该把度数算两遍。map(tuple) 是把内层列表转成可哈希的元组，set 才能去重。
- `next((i for i in ...), -1)`：在 1..n 里找第一个满足"入度 n-1 且出度 0"的人；生成器一个都没产出时，next 返回默认值 -1。这等价于"找到就返回，没找到返回 -1"的一行写法。
- 为什么不用建图搜索：法官条件完全由两个数字决定，扫一遍 trust 数完度数，答案就出来了——建模阶段想清楚"要什么信息"，常常能把整个图搜索都省掉。

## 5. 分段实现：按顺序运行

**本章接口：** `build_adj_list(n, edges, directed)`、`adj_matrix_to_edges(matrix)`、`find_judge(n, trust)`

### build_adj_list

In [1]:
def build_adj_list(n, edges, directed=False):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        if not (0 <= u < n and 0 <= v < n):
            raise ValueError('vertex outside 0..n-1')
        adj[u].append(v)
        if not directed and u != v:
            adj[v].append(u)
    return adj

### adj_matrix_to_edges

In [2]:
def adj_matrix_to_edges(matrix, directed=False):
    n = len(matrix)
    if any((len(row) != n for row in matrix)):
        raise ValueError('square matrix required')
    if not directed and any((bool(matrix[i][j]) != bool(matrix[j][i]) for i in range(n) for j in range(n))):
        raise ValueError('undirected adjacency matrix must be symmetric')
    return [(i, j) for i in range(n) for j in range(n) if matrix[i][j] and (directed or i <= j)]

### find_judge

In [3]:
def find_judge(n, trust):
    incoming = [0] * (n + 1)
    outgoing = [0] * (n + 1)
    for u, v in set(map(tuple, trust)):
        outgoing[u] += 1
        incoming[v] += 1
    return next((i for i in range(1, n + 1) if incoming[i] == n - 1 and outgoing[i] == 0), -1)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

edges=[(0,1),(1,2)]; adj=build_adj_list(4,edges,False)
show_table(['顶点','邻接顶点','度数'],[(i,a,len(a)) for i,a in enumerate(adj)])

顶点,邻接顶点,度数
0,[1],1
1,"[0, 2]",2
2,[1],1
3,[],0


## 7. 正确性、适用条件与复杂度

顶点集合独立于边集合创建，所以没有边的点也不会丢失。法官条件等价于“其他所有人都信任它，且它不信任任何人”，分别由入度n−1与出度0表达。

**代价：** 邻接表O(V+E)构建与存储；矩阵转换O(V²)扫描；法官O(n+E)平均时间。法官标签为1..n，与一般图0..n−1不同。

### 展开理解

**为什么孤立点不会丢：** 顶点的空列表在加边**之前**就全部创建好了，边只是往这些列表里追加内容。一个点哪怕一条边都不沾，它的空列表也稳稳待在 adj 里，后面的遍历算法依然会"路过"它。

**为什么无向要双向存：** 无向边 (u,v) 的含义是"u 能到 v 且 v 能到 u"。搜索算法只会顺着 adj[u] 找邻居，如果只存了 adj[u] 里的 v，那么从 v 出发就找不到 u，等于把无向边偷偷变成了有向边，答案自然错。

**为什么只取上三角：** 无向矩阵里 matrix[i][j] 和 matrix[j][i] 描述的是同一条边。i<=j 时每对 (i,j) 恰好被筛中一次（i>j 的情况被它的镜像 (j,i) 代表了），所以输出不重也不漏；对角线 i==j 不排除，是为了自环不被吞掉。

**法官条件为什么就是入度 n-1、出度 0：** "其他所有人都信任他"翻译过来就是被 n-1 个不同的人信任（入度 n-1）；"他不信任任何人"就是出度 0。反过来，两个数字都满足的人也必然满足法官定义，所以条件和定义等价，数数即得解。

**复杂度（用具体数字感受）：**

- 邻接表：时间和空间都是 O(V+E)。1 万个点、5 万条边的图，就是大约 6 万次列表操作，瞬间完成。
- 矩阵转边表：O(V²)，因为要扫完整个矩阵。1000×1000 的矩阵是 100 万格，还好；10 万×10 万就是 100 亿格，内存直接爆炸——这就是稀疏图要用邻接表的原因。
- 法官：O(n+E)（去重集合那步平均也是这个量级）。n=1000、trust 有 1 万条，也就万级操作。

## 8. 单元测试：每个用例在检查什么

```python
assert build_adj_list(4, [(0, 1)], False) == [[1], [0], [], []]
```
正常值 + **孤立点检查**：4 个点只有 1 条边，结果末尾必须有两个空列表。如果实现时"边加完才建点"，孤立点 2、3 就会消失，这条立即失败。

```python
assert build_adj_list(3, [(0, 1)], True) == [[1], [], []]
```
有向模式：只有 adj[0] 里有 1，adj[1] 里不能有 0。测"方向"这个开关是否真的生效。

```python
assert adj_matrix_to_edges([[0,1,0],[1,0,1],[0,1,0]])==[(0,1),(1,2)])
```
无向对称矩阵：对称位置 (1,0) 和 (2,1) 不允许再产出重复边。预期值恰好 2 条边，专门卡"重复输出"这个 bug。

```python
assert find_judge(2, [[1, 2]]) == 2
```
最简正常值：2 号被 1 信任且不信任任何人。

```python
assert find_judge(1, []) == 1
```
**边界值**：只有 1 个人、没有任何信任记录。这 1 个人"被其他 n-1=0 个人信任"（入度 0=n-1）且出度 0，所以他自己就是法官。这条专门抓"忘了 n=1 时 n-1=0"的边角错误。

```python
assert find_judge(3, [[1, 3], [2, 3], [3, 1]]) == -1
```
**特殊值**：3 号入度达标但出度不为 0（他信任了 1）。测"两个条件必须同时满足"，只查入度的实现会错误返回 3。

In [5]:
assert build_adj_list(4, [(0, 1)], False) == [[1], [0], [], []]

assert build_adj_list(3, [(0, 1)], True) == [[1], [], []]

assert adj_matrix_to_edges([[0, 1, 0], [1, 0, 1], [0, 1, 0]]) == [(0, 1), (1, 2)]

assert find_judge(2, [[1, 2]]) == 2

assert find_judge(1, []) == 1

assert find_judge(3, [[1, 3], [2, 3], [3, 1]]) == -1

print('N079: 所有本章断言通过')

N079: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 把棋盘移动规则建模为边。

**练习 2：** 比较稀疏图与稠密图的存储成本。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（把棋盘移动规则建模为边）：** 思路方向：格子（或坐标）是顶点，"一步能走到"是边。以马走日为例，从 (r,c) 出发有至多 8 个目标格 (r±1,c±2)、(r±2,c±1)，每一个都是一条有向边。提示：顶点编号可以用 `r*列数+c` 把二维坐标压成一维；边界是目标格越出棋盘时要丢弃。手算示例可以先画 3×3 棋盘，列出中心格出发的所有边，再用 build_adj_list（directed=True）验证。

**练习 2（稀疏图 vs 稠密图的存储成本）：** 思路方向：算两笔账。邻接表空间约 O(V+E)；邻接矩阵恒为 O(V²)。取 V=1000：完全图（边约 50 万条）两者都得上百万单位；但随机稀疏图（边只有 2000 条）邻接表约 3000 个槽位，矩阵仍要 100 万格。提示：用 `sys.getsizeof` 或直接数列表元素个数来"手算示例"，把数字列成表；边界情况是 E=0（只有孤立点）和 E=V(V-1)/2（完全图）两个极端。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def build_adj_list(n, edges, directed=False):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        if not (0 <= u < n and 0 <= v < n):
            raise ValueError('vertex outside 0..n-1')
        adj[u].append(v)
        if not directed and u != v:
            adj[v].append(u)
    return adj

def adj_matrix_to_edges(matrix, directed=False):
    n = len(matrix)
    if any((len(row) != n for row in matrix)):
        raise ValueError('square matrix required')
    if not directed and any((bool(matrix[i][j]) != bool(matrix[j][i]) for i in range(n) for j in range(n))):
        raise ValueError('undirected adjacency matrix must be symmetric')
    return [(i, j) for i in range(n) for j in range(n) if matrix[i][j] and (directed or i <= j)]

def find_judge(n, trust):
    incoming = [0] * (n + 1)
    outgoing = [0] * (n + 1)
    for u, v in set(map(tuple, trust)):
        outgoing[u] += 1
        incoming[v] += 1
    return next((i for i in range(1, n + 1) if incoming[i] == n - 1 and outgoing[i] == 0), -1)

# 示例与回归测试
assert build_adj_list(4, [(0, 1)], False) == [[1], [0], [], []]

assert build_adj_list(3, [(0, 1)], True) == [[1], [], []]

assert adj_matrix_to_edges([[0, 1, 0], [1, 0, 1], [0, 1, 0]]) == [(0, 1), (1, 2)]

assert find_judge(2, [[1, 2]]) == 2

assert find_judge(1, []) == 1

assert find_judge(3, [[1, 3], [2, 3], [3, 1]]) == -1

print('N079: 所有本章断言通过')

N079: 所有本章断言通过


## 11. 代表题与迁移

- **997. Find the Town Judge（warmup）：** 就是本章的 find_judge，练"不建图、只数入度出度"的建模眼光。
- **1971. Find if Path Exists in Graph（preview）：** 需要先按本章方法把边列表建成邻接表，之后用 DFS/BFS 判连通——它是下一章 N080 内容的预告，建模部分正是本章的用武之地。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。